# CEIT Corrosion Monitoring Workflow

This notebook walks through the `CeitCorrosionMonitoring` workflow as *explicit, auditable notebook steps* instead of hiding the SHM lookup, POST, and GET calls behind a helper function.

**What this notebook covers**

- **Import** the required SDK components from the local multi-repo workspace.
- **Configure** the JSON file path, API root, CEIT metadata, and token source.
- **Map** each CEIT `sensor_identifier` to the existing SHM sensors found by serial-number pattern matching.
- **Prepare** validated CEIT measurements and derive one stable result series per sensor and metric.
- **Upload** the results into one shared backend analysis named `CeitCorrosionMonitoring`.
- **Retrieve** persisted rows back from the API and reconstruct the normalized CEIT frame.
- **Plot** the retrieved data through the CEIT pyecharts dropdown plot.

**How to use it**

- Run the notebook *top to bottom* the first time.
- Inspect the Mermaid diagrams before each stage to understand *inputs, transformations, and outputs*.
- Use the SHM sensor mapping table as a quick visual check before the shared upload step.

## Mermaid Color Legend

The workflow diagrams in this notebook use a consistent visual language.

- <span style="color:#2B6F77;"><strong>Blue nodes</strong></span>: API calls, services, or external system interactions.
- <span style="color:#5E8C61;"><strong>Green nodes</strong></span>: validated outputs, identifiers, or reconstructed result frames.
- <span style="color:#C08B3E;"><strong>Amber nodes</strong></span>: transformation, serialization, or intermediate processing steps.
- <span style="color:#C47A2C;"><strong>Orange decision/request nodes</strong></span>: branching logic, runtime choices, or fail-fast validation.
- <span style="color:#365A73;"><strong>Blue-grey connectors</strong></span>: data flow between notebook steps.

*Read the diagrams from top to bottom to follow the lifecycle from raw CEIT JSON measurements to plotted backend results.*

## Imports

This section bootstraps the notebook so it can import the core SDK, the results SDK, and the SHM SDK directly from the current multi-repo workspace.

**Why this is explicit**

- The notebook is meant to be runnable from the repository checkout without relying on a separately installed editable environment.
- Import resolution is made visible up front so there is no hidden state about where `owi.metadatabase.*` modules come from.
- The imported APIs stay close to the 1.0 notebook style, with the only extra piece being the SHM client required for sensor resolution.

In [23]:
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [24]:
from pathlib import Path
from typing import Union

import pandas as pd
from IPython.display import display
from owi.metadatabase._utils.utils import load_token_from_env_file
from owi.metadatabase.geometry.io import GeometryAPI
from owi.metadatabase.locations.io import LocationsAPI
from owi.metadatabase.shm import ShmAPI
from rich import print

from owi.metadatabase.results import CeitResultsService, ResultsAPI, load_ceit_measurements
from owi.metadatabase.results.analyses import CEIT_METRICS, ceit_frame_from_measurements


## Constants

This section defines the *runtime configuration* for the notebook.

**Review these values before execution**

- **`DATA_FILE`**: source JSON file containing the raw CEIT measurements.
- **`BASE_URL`**: API root used for locations, geometry, results, and SHM calls.
- **`PROJECTSITE`**, **`ASSETLOCATION`**, and **`MODEL_DEFINITION`**: metadata context used to resolve the correct backend objects.
- **`ANALYSIS_NAME`**: the shared analysis reused or created by this notebook.
- **`TOKEN`**: authentication token loaded from the repository environment file.

*Outcome:* the notebook has one explicit place where environment-specific values can be checked or changed before any API request is made.*

<html>
    <style>
        table {
            border: none;
            border-color: transparent;
            border-spacing: 2px;
            border-collapse: separate;
            border-radius: 8px;
            font-family: "Latin Modern Mono";
        }
    </style>
</html>


In [25]:

# ** Configuration of the script: identifiers, file paths, API roots, etc.

WORKSPACE_ROOT = Path().resolve().parent
DATA_FILE = WORKSPACE_ROOT / "scripts" / "data" / "MeasFile_24sea.json"
DEFAULT_ENV_FILE = WORKSPACE_ROOT / ".env"
TOKEN_ENV_VAR = "OWI_METADATABASE_API_TOKEN"
DEFAULT_RESULTS_API_ROOT = "https://owimetadatabase-dev.azurewebsites.net/api/v1"

BASE_URL = DEFAULT_RESULTS_API_ROOT
PROJECTSITE = "Willow"
ASSETLOCATION = "CEIT"
MODEL_DEFINITION = "CEIT Willow"
ANALYSIS_NAME = "CeitCorrosionMonitoring"
TOKEN = load_token_from_env_file(DEFAULT_ENV_FILE, TOKEN_ENV_VAR)

print("[bold blue]Configuration[/bold blue]")
display(pd.DataFrame([{"data_file": str(DATA_FILE),
                       "api_root": BASE_URL,
                       "projectsite": PROJECTSITE,
                       "assetlocation": ASSETLOCATION,
                       "model_definition": MODEL_DEFINITION,
                       "analysis_name": ANALYSIS_NAME,
                       "token_available": TOKEN is not None}]).T.rename(columns={0: "value"}))


Configuration

,value
data_file,/home/pietro.dantuono@24SEA.local/Projects/SMA...
api_root,https://owimetadatabase-dev.azurewebsites.net/...
projectsite,Willow
assetlocation,CEIT
model_definition,CEIT Willow
analysis_name,CeitCorrosionMonitoring
token_available,True


## Resolve CEIT Metadata

Before looking at the measurements, the notebook resolves the backend metadata context for the shared analysis.

**What is resolved here**

- **`site_id`**: The CEIT project site identifier.
- **`location_id`**: The unique CEIT asset location used to persist location-scoped result rows.
- **`model_definition_id`**: The CEIT Willow model definition identifier.
- **`existing_analysis_id`**: The current `CeitCorrosionMonitoring` analysis row already present in the results backend.

*Outcome:* later upload and retrieval cells can work with explicit ids instead of repeating title-based lookups.


```mermaid
%%{init: {'theme': 'base', 'themeVariables': {
  'fontSize': 'small',
  'primaryColor': '#E8F3F1',
  'primaryTextColor': '#12343B',
  'primaryBorderColor': '#2C6E6F',
  'lineColor': '#365A73',
  'tertiaryColor': '#F6F1E8',
  'clusterBkg': '#F7FAFC',
  'clusterBorder': '#B8C4CF'
}}}%%
graph TD
    A["BASE_URL + TOKEN"] --> B["locations_api = LocationsAPI(...)"]
    A --> C["geometry_api = GeometryAPI(...)"]
    A --> D["results_api = ResultsAPI(...)"]
    B --> E["site_detail = locations_api.get_projectsite_detail(projectsite=PROJECTSITE)"]
    E --> F["site_id"]
    B --> G["willow_locations = locations_api.get_assetlocations(projectsite=PROJECTSITE)['data']"]
    G --> H["ceit_location"]
    H --> I["location_id"]
    C --> J["model_definition_id = geometry_api.get_modeldefinition_id(projectsite=PROJECTSITE, model_definition=MODEL_DEFINITION)['id']"]
    D --> K["existing_analysis_rows = results_api.list_analyses(name=ANALYSIS_NAME)['data']"]
    K --> L["existing_analysis_id"]
    D --> M["service = CeitResultsService(...)"]
    I --> M
    J --> M

    classDef api fill:#D9EEF2,stroke:#2B6F77,color:#16323A;
    classDef transform fill:#F8ECD9,stroke:#C08B3E,color:#50310E;
    classDef output fill:#EAF4E1,stroke:#5E8C61,color:#183A1D;

    class B,C,D,E,G,J,K,M api;
    class A transform;
    class F,H,I,L output;
```


In [26]:

# ** Class instances for API access and CEIT results management

locations_api = LocationsAPI(api_root=BASE_URL, token=TOKEN)
geometry_api = GeometryAPI(api_root=BASE_URL, token=TOKEN)
results_api = ResultsAPI(api_root=BASE_URL, token=TOKEN)
shm_api = ShmAPI(api_root=BASE_URL, token=TOKEN)

# -- site_id
site_detail = locations_api.get_projectsite_detail(projectsite=PROJECTSITE)
site_id = int(site_detail["id"])

# -- location_id
willow_locations = locations_api.get_assetlocations(projectsite=PROJECTSITE)["data"]
ceit_location = willow_locations.loc[
    willow_locations["title"].astype(str).str.casefold() == ASSETLOCATION.casefold()
].copy()
location_id = int(ceit_location.iloc[0]["id"])

# -- model_definition_id
model_definition_id = int(
    geometry_api.get_modeldefinition_id(
        projectsite=PROJECTSITE,
        model_definition=MODEL_DEFINITION,
    )["id"]
)

# -- analysis_id (if existing)
existing_analysis_rows = results_api.list_analyses(name=ANALYSIS_NAME)["data"]
existing_analysis_id = int(existing_analysis_rows.iloc[0]["id"])

# ** Results Service
# It will be used in the next cells to upload and retrieve CEIT results,
# but we can already instantiate it here with all the required identifiers
# resolved.
service = CeitResultsService(api=results_api,
                             model_definition_id=model_definition_id,
                             location_id=location_id)

metadata_summary = pd.DataFrame([{"site_id": site_id,
                                  "location_id": location_id,
                                  "model_definition_id": model_definition_id,
                                  "existing_analysis_id": existing_analysis_id,}]) \
                                 .T.rename(columns={0: "value"})
print("[bold green]Resolved metadata identifiers:[/bold green]")
display(metadata_summary)


Resolved metadata identifiers:

,value
site_id,77
location_id,3443
model_definition_id,18
existing_analysis_id,58


## Load and Normalize the CEIT Measurements

This stage validates the JSON file through the CEIT analysis helpers and expands each raw measurement into one long-form row per metric.

**What happens here**

- The raw JSON file is parsed through `load_ceit_measurements(...)`.
- Each measurement row becomes five normalized observations through `ceit_frame_from_measurements(...)`.
- The notebook derives the unique sensor identifiers that are reused in the SHM lookup table and the shared upload.

*Outcome:* the raw CEIT export is converted into a deterministic input frame for sensor lookup, upload, retrieval, and plotting.*

```mermaid
%%{init: {'theme': 'base', 'themeVariables': {
  'fontSize': 'small',
  'primaryColor': '#E8F3F1',
  'primaryTextColor': '#12343B',
  'primaryBorderColor': '#2C6E6F',
  'lineColor': '#365A73',
  'tertiaryColor': '#F6F1E8',
  'clusterBkg': '#F7FAFC',
  'clusterBorder': '#B8C4CF'
}}}%%
graph TD
    A["DATA_FILE"] --> B["measurements = load_ceit_measurements(DATA_FILE)"]
    B --> C["measurement_frame = ceit_frame_from_measurements(measurements)"]
    B --> D["unique_sensors"]
    B --> E["expected_observation_count"]

    classDef transform fill:#F8ECD9,stroke:#C08B3E,color:#50310E;
    classDef output fill:#EAF4E1,stroke:#5E8C61,color:#183A1D;

    class A,B transform;
    class C,D,E output;
```

In [27]:

# ** Load CEIT measurements

measurements = load_ceit_measurements(DATA_FILE)
measurement_frame = ceit_frame_from_measurements(measurements)
unique_sensors = sorted({measurement.sensor_identifier for measurement in measurements})
expected_observation_count = len(measurements) * len(CEIT_METRICS)

measurement_summary = pd.DataFrame([{"raw_measurement_rows": len(measurements),
                                     "unique_sensors": len(unique_sensors),
                                     "normalized_observations": len(measurement_frame),
                                     "expected_observations": expected_observation_count}]) \
                                   .T.rename(columns={0: "value"})
print("[bold green]Loaded CEIT measurements[/bold green]")
display(measurement_summary)
print("[blue]• Sample of normalized measurement frame[/blue]")
display(measurement_frame.head())
print("[blue]• Observation count by sensor and metric[/blue]")
display(measurement_frame.groupby(["sensor_identifier", "metric"]) \
                         .size().rename("points").reset_index() \
                         .sort_values(["sensor_identifier", "metric"]) \
                         .reset_index(drop=True))


Loaded CEIT measurements

,value
raw_measurement_rows,24
unique_sensors,3
normalized_observations,120
expected_observations,120


• Sample of normalized measurement frame

,sensor_identifier,timestamp,metric,value
0,DA8F,2026-03-12T12:05:12+00:00,temperature,-40.84765
1,DA8F,2026-03-12T12:05:12+00:00,battery,4.13714
2,DA8F,2026-03-12T12:05:12+00:00,tof,1653.43945
3,DA8F,2026-03-12T12:05:12+00:00,amplitude,43.00000
4,DA8F,2026-03-12T12:05:12+00:00,meas_gain,1.39999


• Observation count by sensor and metric

,sensor_identifier,metric,points
0,DA87,amplitude,6
1,DA87,battery,6
2,DA87,meas_gain,6
3,DA87,temperature,6
4,DA87,tof,6
5,DA8F,amplitude,12
6,DA8F,battery,12
7,DA8F,meas_gain,12
8,DA8F,temperature,12
9,DA8F,tof,12


## Resolve SHM Sensors

This stage builds a lightweight lookup table from each CEIT `sensor_identifier` to the SHM sensors whose `serial_number` matches the identifier pattern.

**What happens here**

- Load the current SHM sensor catalog once through `shm_api.list_sensors()`.
- For each CEIT identifier, keep SHM sensors whose `serial_number` contains `<sensor_identifier>-`.
- Collect the matched SHM sensor ids and serial numbers into one display table.
- Use the table as a simple inspection aid before upload.

*Outcome:* the notebook exposes a compact `sensor_identifier_to_id_map` table with no extra control flow around the upload step.*

```mermaid
%%{init: {'theme': 'base', 'themeVariables': {
  'fontSize': 'small',
  'primaryColor': '#E8F3F1',
  'primaryTextColor': '#12343B',
  'primaryBorderColor': '#2C6E6F',
  'lineColor': '#365A73',
  'tertiaryColor': '#F6F1E8',
  'clusterBkg': '#F7FAFC',
  'clusterBorder': '#B8C4CF'
}}}%%
graph TD
    A["shm_api.list_sensors()['data']"] --> B["sensor_frame"]
    C["unique_sensors"] --> D["for sensor_identifier in unique_sensors"]
    B --> E["candidates = _resolve_sensors(sensor_frame, sensor_identifier)"]
    D --> E
    E --> F["matched_sensor_ids + matched_serial_numbers"]
    F --> G["resolution_rows"]
    G --> H["sensor_identifier_to_id_map"]
    H --> I["display(sensor_identifier_to_id_map)"]

    classDef api fill:#D9EEF2,stroke:#2B6F77,color:#16323A;
    classDef transform fill:#F8ECD9,stroke:#C08B3E,color:#50310E;
    classDef output fill:#EAF4E1,stroke:#5E8C61,color:#183A1D;

    class A api;
    class D,E,F,G transform;
    class B,C,H,I output;
```

In [28]:
sensor_frame: pd.DataFrame = shm_api.list_sensors()["data"]

def _resolve_sensors(
    sensor_frame: pd.DataFrame, sensor_identifier: str
) -> pd.DataFrame:
    """Resolve sensors based on the given sensor identifier."""
    serial_numbers = sensor_frame["serial_number"].astype(str)
    matches = sensor_frame.loc[
        serial_numbers.str.contains(f"{sensor_identifier}-", na=False)
    ].copy().drop_duplicates(subset=["id"]).sort_values(["serial_number", "id"])  # ty:ignore[no-matching-overload]
    return matches.reset_index(drop=True)

resolution_rows: list[dict[str, int | str]] = []  # ty:ignore

for sensor_identifier in unique_sensors:
    candidates = _resolve_sensors(sensor_frame, sensor_identifier)
    matched_sensor_ids = [] if candidates.empty else [
        int(value) for value in candidates["id"].tolist()
    ]
    matched_serial_numbers = [] if candidates.empty else [
        str(value) for value in candidates["serial_number"].tolist()
    ]

    resolution_rows.append({"sensor_identifier": sensor_identifier,
                            "matched_sensor_count": len(matched_sensor_ids),
                            "matched_sensor_ids": ", ".join(str(value) for value in matched_sensor_ids),
                            "matched_serial_numbers": ", ".join(matched_serial_numbers),}
    )

sensor_identifier_to_id_map = pd.DataFrame(resolution_rows).sort_values(
    "sensor_identifier"
).reset_index(drop=True)
display(sensor_identifier_to_id_map)


,sensor_identifier,matched_sensor_count,matched_sensor_ids,matched_serial_numbers
0,DA87,1,92,DA87-06
1,DA8F,1,90,DA8F-01
2,DA9D,1,91,DA9D-05


## Build and Upload the Shared Analysis

This section turns the CEIT measurements into backend payloads and writes them into one shared analysis named `CeitCorrosionMonitoring`.

**What happens here**

- Build the shared `AnalysisDefinition` used for the upload.
- Group measurements by `sensor_identifier`.
- Build a minimal `additional_data_by_sensor` payload with the input file name.
- Preview the serialized result payloads that will be created per sensor and metric.
- Upsert each stable sensor-metric series into the shared analysis.

*Outcome:* one shared analysis stores all CEIT sensors, and the upload summary shows how many series were created or updated.*

```mermaid
%%{init: {'theme': 'base', 'themeVariables': {
  'fontSize': 'small',
  'primaryColor': '#E8F3F1',
  'primaryTextColor': '#12343B',
  'primaryBorderColor': '#2C6E6F',
  'lineColor': '#365A73',
  'tertiaryColor': '#F6F1E8',
  'clusterBkg': '#F7FAFC',
  'clusterBorder': '#B8C4CF'
}}}%%
graph TD
    A["ANALYSIS_NAME + DATA_FILE"] --> B["analysis_definition = service.build_shared_analysis_definition(...)"]
    B --> C["analysis_payload"]
    D["measurements_by_sensor + additional_data_by_sensor"] --> E["preview_series = service.build_sensor_results(...)"]
    E --> F["result_payload_preview"]
    G["DATA_FILE + site_id + location_id + ANALYSIS_NAME + additional_data_by_sensor"] --> H["upload_result = service.upsert_measurements_to_shared_analysis(...)"]
    H --> I["analysis_id + analysis_created"]
    H --> J["upload_summary"]

    classDef api fill:#D9EEF2,stroke:#2B6F77,color:#16323A;
    classDef transform fill:#F8ECD9,stroke:#C08B3E,color:#50310E;
    classDef output fill:#EAF4E1,stroke:#5E8C61,color:#183A1D;

    class B,E,H api;
    class A,D,G transform;
    class C,F,I,J output;
```

In [29]:
analysis_definition = service.build_shared_analysis_definition(
    analysis_name=ANALYSIS_NAME,
    source=str(DATA_FILE),
)
analysis_payload = service.analysis_serializer.to_payload(analysis_definition)
print(analysis_payload)


{
    'name': 'CeitCorrosionMonitoring',
    'model_definition_id': 18,
    'location_id': 3443,
    'source_type': 'ceit-json',
    'source': 
'/home/pietro.dantuono@24SEA.local/Projects/SMARTLIFE/OWI-metadatabase-SDK/owi-metadatabase-results-sdk/scripts/dat
a/MeasFile_24sea.json',
    'description': 'Shared CEIT corrosion monitoring analysis across sensors.',
    'additional_data': {'analysis_family': 'CEITSensor', 'shared_analysis': True}
}

In [34]:
analysis_definition = service.build_shared_analysis_definition(
    analysis_name=ANALYSIS_NAME,
    source=str(DATA_FILE),
)
analysis_payload = service.analysis_serializer.to_payload(analysis_definition)

measurements_by_sensor = {sensor_identifier: [measurement
                                              for measurement in measurements
                                              if measurement.sensor_identifier == sensor_identifier]
                          for sensor_identifier in unique_sensors}
additional_data_by_sensor = {sensor_identifier: {"input_file": DATA_FILE.name}
                             for sensor_identifier in unique_sensors}

preview_series = []
for sensor_identifier in unique_sensors:
    preview_series.extend(
        service.build_sensor_results(
            sensor_identifier,
            measurements_by_sensor[sensor_identifier],
            site_id=site_id,
            location_id=location_id,
            analysis_name=ANALYSIS_NAME,
            use_stable_series_keys=True,
            additional_data=additional_data_by_sensor[sensor_identifier],
        )
    )
result_payload_preview = [service.result_serializer.to_payload(series, analysis_id=0)
                          for series in preview_series]

upload_result = service.upsert_measurements_to_shared_analysis(
    DATA_FILE,
    site_id=site_id,
    location_id=location_id,
    analysis_name=ANALYSIS_NAME,
    additional_data_by_sensor=additional_data_by_sensor,
)
analysis_id = int(upload_result["analysis_id"])
analysis_created = bool(upload_result["analysis_created"])

upload_summary = pd.DataFrame(upload_result["summary"])
if not upload_summary.empty:
    upload_summary = upload_summary.sort_values(
        ["sensor_identifier", "metric", "series_key"]
    ).reset_index(drop=True)

print("[bold green]Preview of analysis and results payloads[/bold green]")
display(pd.DataFrame([analysis_payload]))
print("[blue]• Sample of results payloads for the first sensor[/blue]")
display(pd.DataFrame(result_payload_preview[:5]))
print("[blue]• Upload summary[/blue]")
display(upload_summary)
print("[blue]• Summary of uploaded series by sensor and metric[/blue]")
display(pd.DataFrame([{"analysis_id": analysis_id,
                       "analysis_created": analysis_created,
                       "series_uploaded": len(upload_summary),
                       "expected_series": len(unique_sensors) * len(CEIT_METRICS)}]) \
                     .T.rename(columns={0: "value"}))


Uploading shared CEIT result series:   0%|          | 0/15 [00:00<?, ?series/s]

Preview of analysis and results payloads

,name,model_definition_id,location_id,source_type,source,description,additional_data
0,CeitCorrosionMonitoring,18,3443,ceit-json,/home/pietro.dantuono@24SEA.local/Projects/SMA...,Shared CEIT corrosion monitoring analysis acro...,"{'analysis_family': 'CEITSensor', 'shared_anal..."


• Sample of results payloads for the first sensor

,analysis,site,location,name_col1,name_col2,units_col1,units_col2,value_col1,value_col2,short_description,description,additional_data
0,0,77,3443,timestamp,temperature,s,degC,"[1773317292.0, 1773317292.0, 1773403692.0, 177...","[-40.84765, -40.84765, -40.84765, -40.84765, -...",DA87:temperature,CEIT temperature time series for sensor DA87.,"{'analysis_kind': 'time_series', 'result_scope..."
1,0,77,3443,timestamp,battery,s,V,"[1773317292.0, 1773317292.0, 1773403692.0, 177...","[4.13714, 4.13714, 4.13714, 4.13714, 4.13714, ...",DA87:battery,CEIT battery time series for sensor DA87.,"{'analysis_kind': 'time_series', 'result_scope..."
2,0,77,3443,timestamp,tof,s,us,"[1773317292.0, 1773317292.0, 1773403692.0, 177...","[1653.35546, 1653.35546, 1653.35546, 1653.3554...",DA87:tof,CEIT tof time series for sensor DA87.,"{'analysis_kind': 'time_series', 'result_scope..."
3,0,77,3443,timestamp,amplitude,s,count,"[1773317292.0, 1773317292.0, 1773403692.0, 177...","[43.0, 43.0, 43.0, 43.0, 43.0, 43.0]",DA87:amplitude,CEIT amplitude time series for sensor DA87.,"{'analysis_kind': 'time_series', 'result_scope..."
4,0,77,3443,timestamp,meas_gain,s,gain,"[1773317292.0, 1773317292.0, 1773403692.0, 177...","[1.39999, 1.39999, 1.39999, 1.39999, 1.39999, ...",DA87:meas_gain,CEIT meas_gain time series for sensor DA87.,"{'analysis_kind': 'time_series', 'result_scope..."


• Upload summary

,sensor_identifier,metric,series_key,analysis_created,action,result_id,appended_points,signal_id
0,DA87,amplitude,DA87:amplitude,False,unchanged,4821,0,None
1,DA87,battery,DA87:battery,False,unchanged,4819,0,None
2,DA87,meas_gain,DA87:meas_gain,False,unchanged,4822,0,None
3,DA87,temperature,DA87:temperature,False,unchanged,4818,0,None
4,DA87,tof,DA87:tof,False,unchanged,4820,0,None
5,DA8F,amplitude,DA8F:amplitude,False,unchanged,4826,0,None
6,DA8F,battery,DA8F:battery,False,unchanged,4824,0,None
7,DA8F,meas_gain,DA8F:meas_gain,False,unchanged,4827,0,None
8,DA8F,temperature,DA8F:temperature,False,unchanged,4823,0,None
9,DA8F,tof,DA8F:tof,False,unchanged,4825,0,None


• Summary of uploaded series by sensor and metric

,value
analysis_id,58
analysis_created,False
series_uploaded,15
expected_series,15


## Retrieve the Persisted Results

This stage reads the stored rows back from the API and reconstructs the normalized CEIT frame from the persisted result series.

**What happens here**

- Fetch the raw backend rows for the selected analysis id.
- Deserialize them into `ResultSeries` objects.
- Reconstruct the normalized long frame from the persisted rows.
- Display each representation for inspection.

*Outcome:* the notebook shows the persisted analysis in raw API form, deserialized series form, and normalized dataframe form.*

```mermaid
%%{init: {'theme': 'base', 'themeVariables': {
  'fontSize': 'small',
  'primaryColor': '#E8F3F1',
  'primaryTextColor': '#12343B',
  'primaryBorderColor': '#2C6E6F',
  'lineColor': '#365A73',
  'tertiaryColor': '#F6F1E8',
  'clusterBkg': '#F7FAFC',
  'clusterBorder': '#B8C4CF'
}}}%%
graph TD
    A["results_api.list_results(analysis=analysis_id)['data']"] --> B["raw_results_frame"]
    B --> C["display(raw_results_frame.head())"]
    D["service.load_backend_series(analysis=analysis_id)"] --> E["retrieved_series"]
    E --> F["display(retrieved_series[:2])"]
    E --> G["retrieved_frame = service.frame_from_series(retrieved_series)"]
    G --> H["display(retrieved_frame.head(2))"]

    classDef api fill:#D9EEF2,stroke:#2B6F77,color:#16323A;
    classDef transform fill:#F8ECD9,stroke:#C08B3E,color:#50310E;
    classDef output fill:#EAF4E1,stroke:#5E8C61,color:#183A1D;

    class A,D api;
    class G transform;
    class B,C,E,F,H output;
```

In [31]:
raw_results_frame = results_api.list_results(analysis=analysis_id)["data"]
print("[bold green]Raw results as retrieved from the API[/bold green]")
display(raw_results_frame.head())
retrieved_series = service.load_backend_series(analysis=analysis_id)
print("[blue]• Raw results converted to ResultsSeries[/blue]")
display(retrieved_series[:2])
retrieved_frame = service.frame_from_series(retrieved_series)
print("[blue]• ResultsSeries converted to DataFrame[/blue]")
display(retrieved_frame.head(2))


Raw results as retrieved from the API

,id,name_col1,name_col2,name_col3,units_col1,units_col2,units_col3,value_col1,value_col2,value_col3,analysis,site,location,short_description,description,related_object,additional_data,slug,visibility,visibility_groups
0,4821,timestamp,amplitude,None,s,count,None,"[1773317292.0, 1773403692.0, 1773576492.0]","[43.0, 43.0, 43.0]",None,58,77,3443,DA87:amplitude,CEIT amplitude time series for sensor DA87.,"{'type': 'shm.signal', 'id': 371}","{'metric': 'amplitude', 'signal_id': 371, 'inp...",willow_ceit_ceitcorrosionmonitoring_58_da87amp...,usergroup,[]
1,4819,timestamp,battery,None,s,V,None,"[1773317292.0, 1773403692.0, 1773576492.0]","[4.13714, 4.13714, 4.13714]",None,58,77,3443,DA87:battery,CEIT battery time series for sensor DA87.,"{'type': 'shm.signal', 'id': 371}","{'metric': 'battery', 'signal_id': 371, 'input...",willow_ceit_ceitcorrosionmonitoring_58_da87bat...,usergroup,[]
2,4822,timestamp,meas_gain,None,s,gain,None,"[1773317292.0, 1773403692.0, 1773576492.0]","[1.39999, 1.39999, 1.39999]",None,58,77,3443,DA87:meas_gain,CEIT meas_gain time series for sensor DA87.,"{'type': 'shm.signal', 'id': 371}","{'metric': 'meas_gain', 'signal_id': 371, 'inp...",willow_ceit_ceitcorrosionmonitoring_58_da87mea...,usergroup,[]
3,4818,timestamp,temperature,None,s,degC,None,"[1773317292.0, 1773403692.0, 1773576492.0]","[-40.84765, -40.84765, -40.84765]",None,58,77,3443,DA87:temperature,CEIT temperature time series for sensor DA87.,"{'type': 'shm.signal', 'id': 371}","{'metric': 'temperature', 'signal_id': 371, 'i...",willow_ceit_ceitcorrosionmonitoring_58_da87tem...,usergroup,[]
4,4820,timestamp,tof,None,s,us,None,"[1773317292.0, 1773403692.0, 1773576492.0]","[1653.35546, 1653.35546, 1653.35546]",None,58,77,3443,DA87:tof,CEIT tof time series for sensor DA87.,"{'type': 'shm.signal', 'id': 371}","{'metric': 'tof', 'signal_id': 371, 'input_fil...",willow_ceit_ceitcorrosionmonitoring_58_da87tof...,usergroup,[]


• Raw results converted to ResultsSeries

[ResultSeries(analysis_name='CeitCorrosionMonitoring', analysis_kind=<AnalysisKind.TIME_SERIES: 'time_series'>, result_scope=<ResultScope.LOCATION: 'location'>, short_description='DA87:amplitude', description='CEIT amplitude time series for sensor DA87.', site_id=77, location_id=3443, related_object=RelatedObject(type='shm.signal', id=371), data_additional={'metric': 'amplitude', 'signal_id': 371, 'input_file': 'MeasFile_24sea.json', 'series_key': 'DA87:amplitude', 'result_scope': 'location', 'source_field': 'Amplitude', 'analysis_kind': 'time_series', 'analysis_name': 'CeitCorrosionMonitoring', 'match_strategy': None, 'signal_history': {'id': 435, 'status': 'ok', 'signal_id': 371, 'check_date': None, 'checked_by': None, 'visibility': 'usergroup', 'description': None, 'sort_timestamp': '1970-01-01T00:00:00+00:00', 'status_approval': 'pending', 'status_priority': 0, 'is_latest_status': True, 'legacy_signal_id': None, 'sensor_record_id': 92, 'visibility_groups': [], 'sensor_serial_number

• ResultsSeries converted to DataFrame

,analysis_name,short_description,sensor_identifier,metric,signal_id,timestamp,value
0,CeitCorrosionMonitoring,DA87:amplitude,DA87,amplitude,371,2026-03-12T12:08:12+00:00,43.0
1,CeitCorrosionMonitoring,DA87:amplitude,DA87,amplitude,371,2026-03-13T12:08:12+00:00,43.0


## Plot the Retrieved Results

The final stage renders the CEIT results from the retrieved backend rows, not from the raw input file.

**What to inspect in the plot**

- The dropdown should expose one entry per sensor identifier.
- Each sensor chart should show the five CEIT metrics as separate time-series lines.
- The plotted data should match the timestamps and values shown in the retrieved dataframe preview.

*Outcome:* the same persisted analysis can be inspected interactively through the CEIT plotting layer exposed by the SDK.*

```mermaid
%%{init: {'theme': 'base', 'themeVariables': {
  'fontSize': 'small',
  'primaryColor': '#E8F3F1',
  'primaryTextColor': '#12343B',
  'primaryBorderColor': '#2C6E6F',
  'lineColor': '#365A73',
  'tertiaryColor': '#F6F1E8',
  'clusterBkg': '#F7FAFC',
  'clusterBorder': '#B8C4CF'
}}}%%
graph TD
    A["analysis_id"] --> B["shared_plot = service.plot_shared_analysis(analysis_id=analysis_id)"]
    B --> C["shared_plot"]
    C --> D["shared_plot.notebook"]
    D --> E["display(shared_plot.notebook)"]

    classDef api fill:#D9EEF2,stroke:#2B6F77,color:#16323A;
    classDef output fill:#EAF4E1,stroke:#5E8C61,color:#183A1D;

    class B api;
    class A,C,D,E output;
```

In [32]:
shared_plot = service.plot_shared_analysis(analysis_id=analysis_id)
display(shared_plot.notebook)


In [33]:
final_summary = pd.DataFrame([{"analysis_id": analysis_id,
                               "analysis_created": analysis_created,
                               "persisted_series": 0 if raw_results_frame.empty \
                                                     else raw_results_frame["short_description"].nunique(),
                               "input_rows": len(measurements),
                               "normalized_input_observations": len(measurement_frame),
                               "retrieved_observations": len(retrieved_frame),
                               "plot_available": shared_plot.notebook is not None}]) \
                             .T.rename(columns={0: "value"})
display(final_summary)


,value
analysis_id,58
analysis_created,False
persisted_series,15
input_rows,24
normalized_input_observations,120
retrieved_observations,60
plot_available,True
